In [22]:
from pathlib import Path
import sys

data_dir = Path("/srv/shared-data/training-datasets/NA-trainings")
print("=" * 70)
print("Datasets mounted at:" + str(data_dir))
print("=" * 70)
src_dir = Path('/home/frank_shan/dev/python/pyapi/src')

sys.path.insert(0, str(src_dir))


Datasets mounted at:/srv/shared-data/training-datasets/NA-trainings


In [23]:
import json
from pathlib import Path
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from tqdm import tqdm

# Define paths
corpus_dir = Path("rag-syn-corpus")
output_dir = Path("rag-eval-goldens")
output_dir.mkdir(exist_ok=True)

print("✓ Imports and paths configured")

✓ Imports and paths configured


In [24]:
# Define output schema for Q&A pair
class QAPair(BaseModel):
    question: str = Field(description="A specific, answerable question based on the content")
    answer: str = Field(description="A concise, accurate answer derived from the content")

# Initialize Ollama with qwen30b
llm = ChatOllama(model="qwen3:30b-instruct", temperature=0.7)

# Create prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert at creating evaluation question-answer pairs for RAG systems.
    
Given a text chunk, generate ONE high-quality question-answer pair that:
- The question should be specific and answerable from the content
- The answer should be concise and accurate
- Focus on key facts, concepts, or procedures mentioned in the text
- Avoid yes/no questions; prefer what/how/why questions

Return your response as JSON with 'question' and 'answer' fields."""),
    ("user", "Text chunk:\n\n{content}\n\nGenerate a question-answer pair:")
])

# Create the chain
parser = JsonOutputParser(pydantic_object=QAPair)
chain = prompt | llm | parser

print("✓ LLM and prompt chain initialized")

✓ LLM and prompt chain initialized


In [25]:
def generate_qa_pair(chunk):
    """Generate a Q&A pair from a single chunk."""
    try:
        result = chain.invoke({"content": chunk["content"]})
        return {
            "input": result["question"],
            "output": result["answer"]
        }
    except Exception as e:
        print(f"Error processing chunk {chunk.get('chunk_id', 'unknown')}: {e}")
        return None

def process_corpus_file(filepath):
    """Process a single corpus file and generate Q&A pairs."""
    print(f"\nProcessing: {filepath.name}")
    
    # Load corpus chunks
    with open(filepath, 'r', encoding='utf-8') as f:
        chunks = json.load(f)
    
    # Generate Q&A pairs for each chunk
    qa_pairs = []
    for chunk in tqdm(chunks, desc="Generating Q&A pairs"):
        qa_pair = generate_qa_pair(chunk)
        if qa_pair:
            qa_pairs.append(qa_pair)
    
    # Save results
    output_file = output_dir / filepath.name
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(qa_pairs, f, indent=4, ensure_ascii=False)
    
    print(f"✓ Saved {len(qa_pairs)} Q&A pairs to {output_file.name}")
    return len(qa_pairs)

print("✓ Processing functions defined")

✓ Processing functions defined


In [26]:
# Test with one chunk
test_file = corpus_dir / "na_2025_tr_day3B.json"
with open(test_file, 'r', encoding='utf-8') as f:
    test_chunks = json.load(f)

# Generate Q&A for first chunk
test_qa = generate_qa_pair(test_chunks[0])
print("Sample Q&A pair:")
print(json.dumps(test_qa, indent=2, ensure_ascii=False))

Sample Q&A pair:
{
  "input": "Why does the management of Building Automation Systems (BAS) differ between China and North America, and what challenge does this present in North America?",
  "output": "In China, BAS design is typically handled by electrical engineers, while in North America, it is usually managed by mechanical engineers. The challenge in North America is that mechanical engineers often lack sufficient understanding of the operational principles of control components, requiring interdisciplinary knowledge spanning both mechanical and electrical engineering."
}


In [27]:
# Process all corpus files
corpus_files = sorted(corpus_dir.glob("*.json"))
print(f"Found {len(corpus_files)} corpus files to process\n")
print("=" * 70)

total_pairs = 0
for filepath in corpus_files:
    count = process_corpus_file(filepath)
    total_pairs += count

print("\n" + "=" * 70)
print(f"✓ Complete! Generated {total_pairs} total Q&A pairs")
print(f"✓ Output directory: {output_dir.absolute()}")

Found 10 corpus files to process


Processing: na_2025_tr_day1A.json


Generating Q&A pairs: 100%|██████████| 130/130 [01:18<00:00,  1.66it/s]


✓ Saved 130 Q&A pairs to na_2025_tr_day1A.json

Processing: na_2025_tr_day1B.json


Generating Q&A pairs:  80%|████████  | 53/66 [00:29<00:07,  1.79it/s]


KeyboardInterrupt: 